Первый взгляд
shape, head(), info(), describe(). Определить типы признаков: числовые, категориальные, текст, даты. Понять, что есть таргет.


Качество данных
Пропуски, дубликаты, константные и почти константные признаки, невалидные значения, утечки (leakage) — признаки, которые «знают» ответ.


Распределения данных

In [4]:
!ls /content/dataset/train.csv

/content/dataset/train.csv


In [3]:
import pandas as pd

In [23]:
df = pd.read_csv("/content/dataset/train.csv")

In [15]:
df.dtypes

,0
image_name,object
detection,object


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22078 entries, 0 to 22077
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   image_name  22078 non-null  object
 1   detection   22078 non-null  object
dtypes: object(2)
memory usage: 345.1+ KB


In [17]:
print("shape:", df.shape)
print("колонки:", df.columns.tolist())
display(df.head(3))

print("NaN в detection:", df["detection"].isna().sum())
print("дубликатов image_name:", df.image_name.duplicated().sum())

shape: (22078, 2)
колонки: ['image_name', 'detection']


,image_name,detection
0,d4b681c636be92a99d2169685169ddce.JPG,4 0.5298713445663452 0.42401960492134094 0.253...
1,af429cc238c384111b2f6e4ace00638c.JPG,6 0.6738590002059937 0.6928213834762573 0.5684...
2,2329fe3d86fa593db201c3897ad74d4c.JPG,1 0.1510416716337204 0.463131308555603 0.23749...


NaN в detection: 0
дубликатов image_name: 0


In [29]:
example_str = "5 0.8464779853820801 0.5842355489730835 0.03125 0.06529530882835388;5 0.7498760223388672 0.5515867471694946 0.015625 0.04040401801466942;5 0.4138140082359314 0.5162336826324463 0.0900300145149231 0.06421305984258652;5 0.4415920078754425 0.6107500791549683 0.18923598527908325 0.17460328340530396;5 0.4501489996910095 0.6672078371047974 0.17857098579406738 0.21176069974899292;5 0.5824649930000305 0.7316020727157593 0.20907700061798096 0.22510801255702972;5 0.804190993309021 0.8376626968383789 0.22048604488372803 0.3246746361255646;5 0.7781500220298767 0.8542567491531372 0.4407240152359009 0.2914865016937256"
for d in example_str.split(";"):
  print(d)
  p = d.split()
  print(p)
  p = [float(x) for x in p]
  print(p)

5 0.8464779853820801 0.5842355489730835 0.03125 0.06529530882835388
['5', '0.8464779853820801', '0.5842355489730835', '0.03125', '0.06529530882835388']
[5.0, 0.8464779853820801, 0.5842355489730835, 0.03125, 0.06529530882835388]
5 0.7498760223388672 0.5515867471694946 0.015625 0.04040401801466942
['5', '0.7498760223388672', '0.5515867471694946', '0.015625', '0.04040401801466942']
[5.0, 0.7498760223388672, 0.5515867471694946, 0.015625, 0.04040401801466942]
5 0.4138140082359314 0.5162336826324463 0.0900300145149231 0.06421305984258652
['5', '0.4138140082359314', '0.5162336826324463', '0.0900300145149231', '0.06421305984258652']
[5.0, 0.4138140082359314, 0.5162336826324463, 0.0900300145149231, 0.06421305984258652]
5 0.4415920078754425 0.6107500791549683 0.18923598527908325 0.17460328340530396
['5', '0.4415920078754425', '0.6107500791549683', '0.18923598527908325', '0.17460328340530396']
[5.0, 0.4415920078754425, 0.6107500791549683, 0.18923598527908325, 0.17460328340530396]
5 0.450148999691

In [31]:
rows, bad = [], []
for _, r in df.iterrows():
    det = r["detection"]
    if pd.isna(det) or not str(det).strip():
        continue
    for d in str(det).split(";"):
        p = d.split()
        if len(p) != 5:
            bad.append((r["image_name"], d))
            continue
        try:
            rows.append([r["image_name"]] + [float(x) for x in p])
        except ValueError:
            bad.append((r["image_name"], d))

ann = pd.DataFrame(rows, columns=["image", "cls", "cx", "cy", "w", "h"])
print(ann.head(2))
ann["cls"] = ann["cls"].astype(int)
print(f"объектов: {len(ann)} | изображений с разметкой: {ann.image.nunique()} "
      f"| битых записей: {len(bad)}")

                                  image  cls        cx        cy         w  \
0  d4b681c636be92a99d2169685169ddce.JPG  4.0  0.529871  0.424020  0.253983   
1  af429cc238c384111b2f6e4ace00638c.JPG  6.0  0.673859  0.692821  0.568452   

          h  
0  0.730392  
1  0.388528  
объектов: 27980 | изображений с разметкой: 22078 | битых записей: 0


In [8]:
ann.head(2)

,image,cls,cx,cy,w,h
0,d4b681c636be92a99d2169685169ddce.JPG,4,0.529871,0.424020,0.253983,0.730392
1,af429cc238c384111b2f6e4ace00638c.JPG,6,0.673859,0.692821,0.568452,0.388528


In [10]:
import matplotlib.pyplot as plt


In [14]:
cls_cnt = ann.cls.value_counts().sort_index()
img_cnt = ann.groupby("cls").image.nunique()
print(cls_cnt)
print(img_cnt)

cls
0      963
1     2067
2      978
3      786
4     2520
5    11593
6     4507
7     1979
8     1840
9      747
Name: count, dtype: int64
cls
0     960
1    1575
2     978
3     774
4    2410
5    8020
6    4219
7     893
8    1582
9     667
Name: image, dtype: int64


In [21]:
print(f"\nдисбаланс макс/мин: {cls_cnt.max() / cls_cnt.min():.1f}x")


дисбаланс макс/мин: 15.5x
